In [0]:
# MAGIC # Schémas explicites — projet Ventes
# MAGIC 
# MAGIC **Pourquoi ce fichier existe**
# MAGIC 
# MAGIC Quand on lit un CSV avec Spark, deux options s'offrent à nous :
# MAGIC 1. `inferSchema=True` : Spark lit tout le fichier une première fois pour deviner les types, puis le relit pour charger les données. Deux passes = plus lent, et le résultat peut changer d'un run à l'autre si les données évoluent (une colonne 100% entière aujourd'hui peut contenir une valeur textuelle demain, et Spark changera silencieusement son inférence).
# MAGIC 2. **Schéma explicite** (`StructType`) : on déclare une fois pour toutes les noms de colonnes et leurs types. Une seule passe de lecture, comportement 100% prévisible, et toute incohérence avec les données réelles remonte comme une erreur claire plutôt qu'un bug silencieux.
# MAGIC 
# MAGIC En entreprise, l'option 2 est quasi systématique dès qu'un pipeline part en production.

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)

In [0]:
# MAGIC **Choix de conception :**
# MAGIC - `date_vente` reste en `StringType` au niveau du schéma de lecture, et non `DateType` directement.
# MAGIC   Pourquoi : si une seule ligne a une date mal formée, `DateType` au moment de la lecture la transforme en `null` silencieusement — impossible de savoir que le problème existe sans creuser. En lisant en `String`, on garde la valeur brute intacte, puis on la caste en `DateType` explicitement dans le notebook silver, avec un contrôle actif du nombre de valeurs qui échouent au cast (voir notebook silver_ventes).
# MAGIC - `quantite` en `IntegerType` : on sait qu'on a injecté volontairement des valeurs négatives (erreurs de saisie) — le type reste correct (un entier négatif est un entier valide), c'est la règle métier en silver qui filtrera ces lignes, pas le schéma.
# MAGIC - `prix_unitaire` et `montant_total` en `DoubleType`, nullable=True car on sait qu'il y a des valeurs manquantes déjà injectées dans les données sources.

schema_ventes = StructType([
    StructField("id_vente", StringType(), nullable=False),
    StructField("date_vente", StringType(), nullable=True),
    StructField("id_client", StringType(), nullable=True),
    StructField("id_produit", StringType(), nullable=True),
    StructField("produit", StringType(), nullable=True),
    StructField("categorie", StringType(), nullable=True),
    StructField("quantite", IntegerType(), nullable=True),
    StructField("prix_unitaire", DoubleType(), nullable=True),
    StructField("montant_total", DoubleType(), nullable=True),
    StructField("canal_vente", StringType(), nullable=True),
    StructField("region", StringType(), nullable=True),
    StructField("id_vendeur", StringType(), nullable=True),
])

In [0]:
# MAGIC Même logique : `date_retour` en `String`, casté plus tard. `montant_rembourse` nullable puisqu'on sait que des valeurs manquent.

schema_retours = StructType([
    StructField("id_retour", StringType(), nullable=False),
    StructField("id_vente", StringType(), nullable=True),
    StructField("date_retour", StringType(), nullable=True),
    StructField("motif_retour", StringType(), nullable=True),
    StructField("montant_rembourse", DoubleType(), nullable=True),
    StructField("statut", StringType(), nullable=True),
])

In [0]:
# MAGIC Plutôt que d'importer `schema_ventes` et `schema_retours` séparément dans chaque notebook (et de devoir modifier chaque import si on ajoute une table plus tard), on expose une seule fonction. Ça simplifie l'import dans les notebooks silver/gold à une seule ligne, quel que soit le nombre de tables gérées par ce fichier à l'avenir.

def get_schema(table_name: str) -> StructType:
    """
    Retourne le schéma explicite associé à une table source.
    
    Args:
        table_name: nom de la table ('ventes' ou 'retours')
    
    Returns:
        StructType correspondant
    
    Raises:
        ValueError: si le nom de table n'est pas reconnu
    """
    schemas = {
        "ventes": schema_ventes,
        "retours": schema_retours,
    }
    if table_name not in schemas:
        raise ValueError(
            f"Aucun schéma défini pour '{table_name}'. "
            f"Tables disponibles : {list(schemas.keys())}"
        )
    return schemas[table_name]